# Votations f?d?rales ? exploration du pipeline PX

Ex?cuter `python -m scripts.run_pipeline` depuis la racine. Ce notebook lit les tables existantes, sans t?l?chargement ni modification des sources.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display

# Fonctionne depuis la racine ou le dossier du notebook.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "environment.yml").is_file())

all_levels = pd.read_parquet(ROOT / "data/interim/ofs-votes/votes_all_levels.parquet")
votes = pd.read_parquet(ROOT / "data/processed/ofs-votes/votes_communes.parquet")
display(votes.head())

## Niveaux territoriaux et couverture

Le cube complet contient aussi les agr?gats nationaux, cantonaux et de district.

In [ ]:
display(all_levels.groupby("level").size().rename("rows"))
display(votes.groupby(votes.vote_date.dt.year).agg(rows=("vote_id", "size"), objects=("vote_id", "nunique"), communes=("municipality_id", "nunique")))

## Valeurs manquantes et contr?les

Les identifiants OFS, et non les noms, d?finissent les cl?s. Les unit?s sp?ciales, questions subsidiaires et r?sultats provisoires sont signal?s dans la source.

In [ ]:
display(all_levels.isna().sum().sort_values(ascending=False))
print("Duplicate keys:", votes.duplicated(["municipality_id", "vote_id"]).sum())
display(votes.filter(regex="^(is_|flag_)").sum())
display(votes[["yes_share", "turnout"]].describe())

Le d?tail du nettoyage est produit dans `reports/cleaning_log.md` et `reports/validation_report.md`. Les anciennes collectes CSV par API ont ?t? remplac?es par le pipeline PX.